In [19]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from string import ascii_uppercase
from geopy.distance import geodesic

from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

In [20]:
X = np.load('../pollenGeolocation_saved/X.npy')
y = np.load('../pollenGeolocation_saved/y.npy')

In [21]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1298)

# Create scaler for X and y
sc_X = StandardScaler()
sc_y = StandardScaler()

# Scale training data
X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)  

# Scale test data using training parameters
X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)  

In [25]:
## WIP
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from xgboost import XGBRegressor
from tqdm import tqdm
import time
from scipy.stats import uniform

def construct_model():
    # the list of regressors to use
    regression_models = [
        linear_model.MultiTaskLasso(),
        MultiOutputRegressor(SVR()),
        KNeighborsRegressor(),
        DecisionTreeRegressor(),
        RandomForestRegressor(),
        MultiOutputRegressor(XGBRegressor())
    ]

    # Define hyperparameter distributions with refined ranges
    lasso_parameters = {
        'alpha': np.logspace(-4, 1, 50)  # Alpha in log scale from 0.0001 to 1
    }

    svr_parameters = {
        'estimator__C': uniform(0.1, 10),  # Uniform distribution between 0.1 and 10
        'estimator__kernel': ['linear', 'rbf', 'poly'],
        'estimator__gamma': ['scale', 'auto'] + list(np.logspace(-3, 3, 50))
    }

    knn_parameters = {
        'n_neighbors': np.arange(2, 50, 1),  # Number of neighbors (2 to 50)
        'weights': ['uniform', 'distance'],  # Weighting method
        'p': [1, 2],  # Manhattan or Euclidean distance
        'algorithm': ['auto']  # Default to 'auto'
    }

    dt_parameters = {
        'max_depth': [None] + list(range(3, 21, 1)),      # Depths from 3 to 20 + unlimited depth
        'min_samples_split': [2, 5, 10, 20, 50],          # Minimum samples to split
        'min_samples_leaf': [1, 2, 5, 10, 20, 50],        # Minimum samples in a leaf
        'max_features': ['auto', 'sqrt', 'log2', None],   # Features considered at each split
        'max_leaf_nodes': [None] + list(range(10, 101, 10))  # Maximum number of leaf nodes
    }

    rf_parameters = {
        'n_estimators': [100, 200, 500, 1000],           # Number of trees
        'max_depth': [None] + list(range(10, 51, 10)),   # Maximum depth of trees
        'min_samples_split': [2, 5, 10, 20],             # Minimum samples to split a node
        'min_samples_leaf': [1, 2, 4, 10, 20],           # Minimum samples in a leaf node
        'max_features': ['auto', 'sqrt', 'log2', None],  # Features considered for splits
        'bootstrap': [True, False],                      # Whether to use bootstrapping
        'max_samples': [0.5, 0.75, None]                 # Optional: Samples per tree if bootstrap=True
    }

    xgb_parameters = {
        'estimator__n_estimators': [100, 200, 500, 1000],          # Number of trees
        'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],  # Learning rate
        'estimator__max_depth': [3, 5, 7, 10, 15, 20],            # Maximum tree depth
        'estimator__min_child_weight': [1, 5, 10, 20],            # Minimum child weight
        'estimator__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],        # Subsampling ratio
        'estimator__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Features per tree
        'estimator__gamma': [0, 0.1, 0.2, 0.5, 1, 5],             # Minimum loss reduction
        'estimator__reg_alpha': [0, 0.01, 0.1, 1, 10, 100],       # L1 regularization
        'estimator__reg_lambda': [1, 10, 50, 100]                 # L2 regularization
    }


    # Combine parameter distributions into a list
    parameters = [
        lasso_parameters,
        svr_parameters,
        knn_parameters,
        dt_parameters,
        rf_parameters,
        xgb_parameters
    ]

    data = {'estimators': []}
    
    # Progress bar setup
    total_models = len(regression_models)
    pbar = tqdm(total=total_models, desc="Optimizing Models", unit="model")
    
    # Iterate through each regressor and use RandomizedSearchCV
    for i, regressor in enumerate(regression_models):
        start_time = time.time()
        print(f"Starting optimization for {regressor.__class__.__name__}...")

        clf = RandomizedSearchCV(
            regressor,                         # Regressor
            param_distributions=parameters[i],  # Hyperparameter distributions
            n_iter=20,                          # Number of random samples to test
            scoring='neg_mean_absolute_error',  # Metric for scoring
            cv=3,                              # 3-fold cross-validation
            n_jobs=-1,                          # Use all available CPU cores
            error_score='raise',                # Raise errors during search
            verbose=1,                          # Verbose output
            random_state=37                     # For reproducibility
        )
        
        clf.fit(X_train, y_train)
        elapsed_time = time.time() - start_time
        
        # Add the fitted model to the estimators list
        data['estimators'].append((regressor.__class__.__name__, clf))
        pbar.set_postfix({"Last Model Time (s)": round(elapsed_time, 2)})
        pbar.update(1)

    pbar.close()
    return data

In [ ]:
## WIP
construct_model()

Optimizing Models:   0%|          | 0/6 [00:00<?, ?model/s]

Starting optimization for MultiTaskLasso...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Optimizing Models:  17%|█▋        | 1/6 [00:06<00:33,  6.73s/model, Last Model Time (s)=6.72]

Starting optimization for MultiOutputRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [14]:
def construct_model():
    # the list of classifiers to use
    regression_models = [
        linear_model.MultiTaskLasso(),
        MultiOutputRegressor(SVR()),
        KNeighborsRegressor(algorithm='auto'),
        DecisionTreeRegressor()
        
    ]

    linear_model_parameters = {}

    lasso_parameters = {
        'alpha': np.arange(0.00, 1.0, 0.01)
    }

    svr_parameters = {
        'estimator__epsilon': np.linspace(0.001, 1, 10),
        'estimator__kernel': ['linear'],
        'estimator__C': np.linspace(0.01, 100, 10)
    }

    knn_parameters = {
        'n_neighbors': np.arange(2, 50, 1),
        'weights': ['uniform', 'distance']
    }

    dt_parameters = {
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

    rf_parameters = {

    }

    parameters = [
        lasso_parameters,
        svr_parameters, #took ~18 hours
        knn_parameters, #took ~min
        dt_parameters, #took ~min
        rf_parameters
    ]
    data['estimators'] = []
    
    # iterate through each classifier and use GridSearchCV
    for i, regressor in enumerate(regression_models):
        clf = GridSearchCV(regressor,              # model
                  param_grid = parameters[i], # hyperparameters
                  scoring='neg_mean_absolute_error',         # metric for scoring
                  cv=10,
                  n_jobs=-1, error_score='raise', verbose=3)
        clf.fit(X_train, y_train)
        # add the clf to the estimators list
        data['estimators'].append((regressor.__class__.__name__, clf))

In [11]:
construct_model()

NameError: name 'data' is not defined

In [13]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for DecisionTreeRegressor: -0.05067355978149486
Best Hyperparameters for DecisionTreeRegressor: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2}




In [9]:
construct_model()

Fitting 10 folds for each of 96 candidates, totalling 960 fits


In [ ]:
#data = {}
construct_model()

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits


In [10]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for KNeighborsRegressor: -0.06883451996121953
Best Hyperparameters for KNeighborsRegressor: {'n_neighbors': 2, 'weights': 'distance'}




In [6]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for MultiTaskLasso: -0.08677049729288025
Best Hyperparameters for MultiTaskLasso: {'alpha': 0.01}


Best Score for MultiOutputRegressor: -0.07225225150444294
Best Hyperparameters for MultiOutputRegressor: {'estimator__C': 0.01, 'estimator__epsilon': 0.001, 'estimator__kernel': 'linear'}




In [ ]:
construct_model()